# 00 — Environment check

Phase 1 deliverable. Imports core stack and logs a no-op run to wandb (offline if not authed).
Run on laptop CPU. Should finish in seconds.

In [ ]:
import platform
import sys

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import sklearn
import torch

import trinetravir

print(f"Python      {sys.version.split()[0]} on {platform.machine()}")
print(f"trinetravir {trinetravir.__version__}")
print(f"scanpy      {sc.__version__}")
print(f"anndata     {ad.__version__}")
print(f"scvi-tools  {scvi.__version__}")
print(
    f"torch       {torch.__version__}  | mps_available={torch.backends.mps.is_available()}  cuda_available={torch.cuda.is_available()}"
)
print(f"sklearn     {sklearn.__version__}")

## Synthetic AnnData round-trip
Confirms scanpy/anndata I/O works end-to-end.

In [ ]:
rng = np.random.default_rng(42)
X = rng.poisson(1.0, size=(100, 50)).astype(np.float32)
obs = pd.DataFrame(
    {
        "virus": rng.choice(["sars_cov_2", "iav", "mock"], size=100),
        "infection_status": rng.choice(["infected", "bystander", "mock"], size=100),
    },
    index=[f"cell_{i}" for i in range(100)],
)
var = pd.DataFrame(index=[f"gene_{j}" for j in range(50)])
adata = ad.AnnData(X=X, obs=obs, var=var)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(adata)

## wandb hello-world
Runs in offline mode by default so no account is required to verify the install.
Once `wandb login` is configured, change `mode="online"` to push real runs.

In [ ]:
import wandb

run = wandb.init(
    project="trinetravir",
    name="00_env_check",
    mode="offline",
    config={"phase": 1, "deliverable": "hello_world"},
)
wandb.log({"n_cells": adata.n_obs, "n_genes": adata.n_vars})
run.finish()
print("wandb offline log written; sync later with: wandb sync wandb/offline-run-*/")